# 01. Exploratory analysis

This notebook is for looking around. It is **not** where the pipeline lives.

Everything here calls into `fraud_pipeline.eda` and `fraud_pipeline.feature_selection`, the same
code the `fraud eda` command runs. Nothing is reimplemented in a cell, because a number computed
in a notebook and a number computed in the pipeline that disagree is the worst outcome available.

Use this to poke at something new. When it turns into a rule, move it into the module and give it
a test.

**The training split only.** The test split is never opened, here or anywhere else before the
evaluation stage. Choosing features by looking at test data would make every metric this project
reports meaningless.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from fraud_pipeline import eda, feature_selection, plots
from fraud_pipeline.config import load_config
from fraud_pipeline.splits import load_split

pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 160)

config = load_config()
train = load_split(config, config.eda.analysis_split)
validation = load_split(config, config.eda.drift_split)

print(f"train      {train.shape}, {train['Class'].sum()} fraud ({train['Class'].mean():.4%})")
print(f"validation {validation.shape}, {validation['Class'].sum()} fraud ({validation['Class'].mean():.4%})")

## The class balance

This is the number that shapes every later decision. At this rate, a model that predicts "normal"
every single time is over 99.8 percent accurate and catches nothing, which is why accuracy appears
nowhere in this project.

In [ ]:
counts = train["Class"].value_counts()
print(counts)
print(f"\nfraud rate       {train['Class'].mean():.5%}")
print(f"always-normal    {1 - train['Class'].mean():.5%} accurate")
print(f"one fraud case is worth {1 / train['Class'].sum():.2%} of recall")

## Per feature statistics

Six measures per feature. They are not redundant:

| Measure | What it asks |
| --- | --- |
| `auc` | does fraud sit consistently high or low on this feature |
| `ks` | do the two distributions differ **in any way at all** |
| `abs_corr` | linear association with the target |
| `mutual_info` | non linear dependence |
| `psi` | how far the feature moved between train and validation |
| `range_coverage` | has the model seen values like the next period's before |

The gap between `auc` and `ks` is the interesting one. A feature can score near chance on AUC
while separating the classes clearly, if fraud clusters in the middle of the range rather than at
one end.

In [ ]:
stats = eda.compute_feature_stats(train, validation, config)
stats

## How much signal is real

Shuffle the target so any relationship is destroyed, then record the best score any feature still
reaches by chance. Repeat, and take a high quantile of those maxima.

Taking the **maximum across all features** on each shuffle is the point. Testing thirty features
and keeping the best is thirty chances to be fooled, and this accounts for that.

Try changing `permutation_shuffles` and watch how stable the ceiling is.

In [ ]:
noise = eda.permutation_noise_ceiling(train, config)
print(noise.describe())

below_auc = stats[stats["auc"] < noise.ceiling]
print(f"\n{len(below_auc)} features fall below the AUC ceiling:")
print(below_auc[["feature", "auc", "ks"]].to_string(index=False))

rescued = below_auc[below_auc["ks"] >= noise.ks_ceiling]
print(f"\nof those, {len(rescued)} still clear the KS ceiling and are kept:")
print(rescued[["feature", "auc", "ks"]].to_string(index=False))

## Leakage and redundancy

Two questions that have to be asked of any dataset before trusting a result.

On this dataset both come back empty, and that is the expected answer rather than a lucky one:
V1 to V28 are principal components, so they are orthogonal to each other by construction and none
of the original columns survive to encode the answer.

The checks still run every time, because the IEEE CIS dataset has real named columns where a leak
is a genuine possibility.

In [ ]:
print("exact duplicate columns:", eda.exact_duplicate_columns(train, config))
print("pairs above the duplicate threshold:", len(eda.correlated_pairs(train, config)))

print("\nmost correlated pairs, whatever their strength:")
print(eda.correlated_pairs(train, config, threshold=0.0).head(10).to_string(index=False))

leaking = stats[stats["auc"] >= config.feature_selection.leakage_auc]
print(f"\nfeatures above the leakage threshold: {len(leaking)}")

## Stability into the next period

`psi` says the distribution moved, which a model can often survive. `range_coverage` says the new
values sit outside anything the model was fitted on, which it cannot.

Watch `Time`.

In [ ]:
print(stats.nlargest(10, "psi")[["feature", "auc", "psi", "range_coverage"]].to_string(index=False))

print(f"\ntrain Time      {train['Time'].min():>10,.0f} to {train['Time'].max():>10,.0f}")
print(f"validation Time {validation['Time'].min():>10,.0f} to {validation['Time'].max():>10,.0f}")
print("\nThe two windows do not overlap at all, which is what a raw timestamp always does.")

## The daily cycle

This is the finding that changed how the split level fraud rates should be read. Folding the
timeline onto a 24 hour clock shows fraud concentrating in the small hours, when transaction
volume is at its lowest.

Validation happens to cover only the afternoon, so most of its lower fraud rate is which hours it
contains rather than fraud behaviour changing.

In [ ]:
clock = (train["Time"] / 3600.0) % 24
by_hour = train.assign(hour=clock.astype(int)).groupby("hour")["Class"].agg(["size", "sum", "mean"])
by_hour["rate_pct"] = by_hour["mean"] * 100
print(by_hour[["size", "sum", "rate_pct"]].to_string())

night = train[clock.between(1, 5)]["Class"].mean()
rest = train[~clock.between(1, 5)]["Class"].mean()
print(f"\n01:00 to 05:00 {night:.4%}")
print(f"other hours    {rest:.4%}")
print(f"ratio          {night / rest:.2f}x")

window = ((validation["Time"] / 3600.0) % 24)
matched = train[clock.between(window.min(), window.max())]["Class"].mean()
print(f"\ntrain over validation's clock window: {matched:.4%}")
print(f"validation itself:                    {validation['Class'].mean():.4%}")
print("Most of the split level gap is the clock, not drift.")

## What the selection system decides

The same call the `fraud eda` command makes. Every drop carries its evidence, so nothing here is
a judgement that cannot be argued with.

In [ ]:
selection = feature_selection.select_features(
    stats,
    eda.correlated_pairs(train, config),
    noise,
    config,
    eda.exact_duplicate_columns(train, config),
)

print(selection.counts())
for decision in selection.decisions:
    if decision.dropped_by_tier is not None:
        print(f"  tier {decision.dropped_by_tier}  {decision.feature:<8} {decision.reasons[0]}")

## Figures

Written to `reports/figures/`. The README uses these directly.

In [ ]:
for path in plots.draw_all(train, validation, stats, noise, config):
    print(path)

## Ideas worth trying

Scratch space. Anything here that proves out gets moved into the module with a test.

- Does an interaction between two weak features beat either alone? Univariate screening cannot see
  this, and it is the known blind spot of the tier 2 rule.
- Do the features that drifted most (V1, V3, V28) also lose the most accuracy on validation?
- Is the amount distribution for fraud different by hour of day, or is it flat?